METRICS DASHBOARD - Real-time visualization
Sleduj performance agenta během běhu a po skončení

In [1]:
import json
import pandas as pd
from pathlib import Path
from datetime import datetime
from typing import Dict, List
import plotly.graph_objects as go
import plotly.express as px

In [2]:
class MetricsDashboard:
    """Vizualizace metrik z agent běhu"""
    
    def __init__(self, metrics_file: str = "agent_metrics.json"):
        self.metrics_file = metrics_file
        self.data = None
        self.load_metrics()
    
    def load_metrics(self):
        """Načti JSON metrics"""
        try:
            with open(self.metrics_file, 'r', encoding='utf-8') as f:
                self.data = json.load(f)
        except FileNotFoundError:
            print(f"⚠️ Metrics file not found: {self.metrics_file}")
            self.data = None
    
    def generate_html_dashboard(self, output_file: str = "dashboard.html"):
        """Generuj HTML dashboard s Plotly grafy"""
        
        if not self.data:
            print("❌ No metrics data to visualize")
            return
        
        # Připrav data
        df_urls = pd.DataFrame(self.data['url_attempts'])
        
        # 1. Success Rate by Website Type
        fig1 = self._create_success_by_type(df_urls)
        
        # 2. Confidence Distribution
        fig2 = self._create_confidence_dist(df_urls)
        
        # 3. Topic Breakdown
        fig3 = self._create_topic_breakdown(df_urls)
        
        # 4. Judge Agreement
        fig4 = self._create_judge_agreement()
        
        # Kombinuj do jednoho HTML
        html_content = f"""
        <!DOCTYPE html>
        <html>
        <head>
            <title>LLM Agent Metrics Dashboard</title>
            <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
            <style>
                body {{
                    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
                    margin: 0;
                    padding: 20px;
                    background: #f5f5f5;
                }}
                .container {{
                    max-width: 1400px;
                    margin: 0 auto;
                }}
                h1 {{
                    color: #1f33a3;
                    border-bottom: 3px solid #3282f6;
                    padding-bottom: 10px;
                }}
                .stats-row {{
                    display: grid;
                    grid-template-columns: repeat(auto-fit, minmax(250px, 1fr));
                    gap: 20px;
                    margin: 30px 0;
                }}
                .stat-card {{
                    background: white;
                    padding: 20px;
                    border-radius: 8px;
                    box-shadow: 0 2px 4px rgba(0,0,0,0.1);
                }}
                .stat-value {{
                    font-size: 32px;
                    font-weight: bold;
                    color: #3282f6;
                }}
                .stat-label {{
                    font-size: 14px;
                    color: #666;
                    margin-top: 8px;
                }}
                .chart-container {{
                    background: white;
                    padding: 20px;
                    border-radius: 8px;
                    box-shadow: 0 2px 4px rgba(0,0,0,0.1);
                    margin: 20px 0;
                }}
                .chart {{
                    width: 100%;
                    height: 400px;
                }}
                .grid-2 {{
                    display: grid;
                    grid-template-columns: 1fr 1fr;
                    gap: 20px;
                }}
                @media (max-width: 1000px) {{
                    .grid-2 {{
                        grid-template-columns: 1fr;
                    }}
                }}
            </style>
        </head>
        <body>
            <div class="container">
                <h1>🤖 LLM Autonomous Agent - Metrics Dashboard</h1>
                
                <div class="stats-row">
                    <div class="stat-card">
                        <div class="stat-value">{self.data['summary']['urls_approved']}</div>
                        <div class="stat-label">URLs Schváleno</div>
                    </div>
                    <div class="stat-card">
                        <div class="stat-value">{self.data['summary']['total_urls_processed']}</div>
                        <div class="stat-label">URLs Zpracováno</div>
                    </div>
                    <div class="stat-card">
                        <div class="stat-value">{self.data['summary']['urls_approved'] / max(self.data['summary']['total_urls_processed'], 1) * 100:.1f}%</div>
                        <div class="stat-label">Success Rate</div>
                    </div>
                    <div class="stat-card">
                        <div class="stat-value">{self.data['summary']['execution_time_seconds']:.0f}s</div>
                        <div class="stat-label">Čas Zpracování</div>
                    </div>
                </div>
                
                <div class="grid-2">
                    <div class="chart-container">
                        <h2>Success Rate by Website Type</h2>
                        <div id="chart1" class="chart"></div>
                    </div>
                    <div class="chart-container">
                        <h2>Audit Confidence Distribution</h2>
                        <div id="chart2" class="chart"></div>
                    </div>
                </div>
                
                <div class="chart-container">
                    <h2>Topic Breakdown</h2>
                    <div id="chart3" style="width: 100%; height: 400px;"></div>
                </div>
                
                {self._render_judge_section()}
                
                <div class="chart-container">
                    <h2>Detailed Results Table</h2>
                    {self._render_table(df_urls)}
                </div>
            </div>
            
            <script>
                {self._render_plotly_js(fig1, fig2, fig3, fig4)}
            </script>
        </body>
        </html>
        """
        
        with open(output_file, 'w', encoding='utf-8') as f:
            f.write(html_content)
        
        print(f"✅ Dashboard generated: {output_file}")
    
    def _create_success_by_type(self, df: pd.DataFrame) -> go.Figure:
        """Graphs: Success rate by website type"""
        
        # Group by website type and verdict
        grouped = df.groupby(['website_type', 'verdict']).size().unstack(fill_value=0)
        
        # Calculate success rate
        success_rates = (grouped.get('APPROVE', 0) / (grouped.sum(axis=1))) * 100
        success_rates = success_rates.sort_values(ascending=False)
        
        fig = go.Figure(
            data=[go.Bar(
                x=success_rates.index,
                y=success_rates.values,
                marker_color='#3282f6',
                text=[f'{v:.1f}%' for v in success_rates.values],
                textposition='auto'
            )]
        )
        
        fig.update_layout(
            title="Success Rate by Website Type",
            xaxis_title="Website Type",
            yaxis_title="Success Rate (%)",
            height=400,
            showlegend=False
        )
        
        return fig
    
    def _create_confidence_dist(self, df: pd.DataFrame) -> go.Figure:
        """Graph: Confidence distribution"""
        
        fig = go.Figure(
            data=[go.Histogram(
                x=df['confidence'],
                nbinsx=20,
                marker_color='#f6a932',
            )]
        )
        
        fig.update_layout(
            title="Audit Confidence Distribution",
            xaxis_title="Confidence Score",
            yaxis_title="Frequency",
            height=400,
            showlegend=False
        )
        
        return fig
    
    def _create_topic_breakdown(self, df: pd.DataFrame) -> go.Figure:
        """Graph: Topic breakdown"""
        
        topic_stats = df.groupby('topic').agg({
            'verdict': lambda x: (x == 'APPROVE').sum()
        }).rename(columns={'verdict': 'approved'})
        
        topic_stats['total'] = df.groupby('topic').size()
        topic_stats['success_rate'] = (topic_stats['approved'] / topic_stats['total']) * 100
        topic_stats = topic_stats.sort_values('success_rate', ascending=False)
        
        fig = go.Figure(
            data=[
                go.Bar(
                    name='Approved',
                    x=topic_stats.index,
                    y=topic_stats['approved'],
                    marker_color='#22c55e'
                ),
                go.Bar(
                    name='Rejected',
                    x=topic_stats.index,
                    y=topic_stats['total'] - topic_stats['approved'],
                    marker_color='#ef4444'
                )
            ]
        )
        
        fig.update_layout(
            title="Results by Topic",
            barmode='stack',
            xaxis_title="Topic",
            yaxis_title="Count",
            height=400
        )
        
        return fig
    
    def _create_judge_agreement(self) -> go.Figure:
        """Graph: Judge agreement analysis"""
        
        if not self.data.get('judge_reviews'):
            return None
        
        df_judge = pd.DataFrame(self.data['judge_reviews'])
        agreement_counts = df_judge['agrees'].value_counts()
        
        fig = go.Figure(
            data=[go.Pie(
                labels=['Agree', 'Disagree'] if len(agreement_counts) == 2 else ['Agree'],
                values=agreement_counts.values,
                marker_colors=['#22c55e', '#ef4444'],
                hole=0.3
            )]
        )
        
        fig.update_layout(
            title="Judge vs Auditor Agreement",
            height=400
        )
        
        return fig
    
    def _render_judge_section(self) -> str:
        """Render judge agreement section"""
        
        if not self.data.get('judge_reviews'):
            return "<p style='color: #666;'>No judge reviews recorded</p>"
        
        df_judge = pd.DataFrame(self.data['judge_reviews'])
        agreement_rate = (df_judge['agrees'].sum() / len(df_judge)) * 100
        
        return f"""
        <div class="chart-container">
            <h2>Judge Agreement Analysis</h2>
            <div id="chart4" class="chart"></div>
            <p style='margin-top: 20px; color: #666;'>
                Judge reviewed {len(df_judge)} marginální rozhodnutí.
                Souhlasil v {agreement_rate:.1f}% případů.
            </p>
        </div>
        """
    
    def _render_table(self, df: pd.DataFrame) -> str:
        """Render HTML table"""
        
        display_cols = ['topic', 'url', 'website_type', 'confidence', 'verdict']
        df_display = df[display_cols].head(20)
        
        html = df_display.to_html(
            index=False,
            classes='results-table',
            float_format=lambda x: f'{x:.2f}'
        )
        
        style = """
        <style>
            .results-table {
                width: 100%;
                border-collapse: collapse;
            }
            .results-table th {
                background: #f0f0f0;
                padding: 12px;
                text-align: left;
                font-weight: 600;
                border-bottom: 2px solid #ddd;
            }
            .results-table td {
                padding: 10px 12px;
                border-bottom: 1px solid #eee;
            }
            .results-table tr:hover {
                background: #f9f9f9;
            }
        </style>
        """
        
        return style + html
    
    def _render_plotly_js(self, fig1, fig2, fig3, fig4) -> str:
        """Render Plotly JS"""
        
        # Simplified - v reálu bys konvertoval Plotly figures na JSON
        return """
        // Plotly charts would be rendered here
        console.log('Metrics dashboard loaded');
        """

# USAGE

In [8]:
import os

current_dir = os.getcwd()
print(f"Current dir: {current_dir}\n")

print("Files here:")
for f in os.listdir('.'):
    if f.endswith(('.py', '.ipynb')):
        print(f"  • {f}")

print(f"\nChecking required files:")
print(f"  autonomous_agent.py: {os.path.exists('autonomous_agent.py')}")
print(f"  metrics_dashboard.py: {os.path.exists('metrics_dashboard.py')}")

# Try import
try:
    from autonomous_agent import WebFinderAgent
    print("\n✅ SUCCESS - Soubory jsou na místě!")
except ModuleNotFoundError:
    print("\n❌ FILES NOT FOUND")
    print("\nRUN IN TERMINAL:")
    print(f"  cd {current_dir}")
    print(f"  cp /path/to/autonomous_agent.py .")
    print(f"  cp /path/to/metrics_dashboard.py .")

Current dir: /root/karel/Discover/Interest focused websites LLM

Files here:
  • Agent Metrics.ipynb
  • LLM website finder.ipynb
  • Agentic website finder.ipynb

Checking required files:
  autonomous_agent.py: False
  metrics_dashboard.py: False

❌ FILES NOT FOUND

RUN IN TERMINAL:
  cd /root/karel/Discover/Interest focused websites LLM
  cp /path/to/autonomous_agent.py .
  cp /path/to/metrics_dashboard.py .


In [10]:
from metrics_dashboard import MetricsDashboard

dashboard = MetricsDashboard('agent_metrics.json')
dashboard.generate_html_dashboard('dashboard.html')
print("✅ dashboard.html ready")

Current dir: /root/karel/Discover/Interest focused websites LLM

Files here:
  • Agent Metrics.ipynb
  • metrics_dashboard.py
  • autonomous_agent.py
  • LLM website finder.ipynb
  • Agentic website finder.ipynb

Checking required files:
  autonomous_agent.py: True
  metrics_dashboard.py: True


Api klíč pro LLM proxy: ········


Files: ['dashboard.html', 'agent_metrics.json', 'Agent Metrics.ipynb', 'metrics_dashboard.py', '__pycache__', 'autonomous_agent.py', '.ipynb_checkpoints', 'LLM website finder.ipynb', 'Agentic website finder.ipynb']
✅ Ready!

✅ SUCCESS - Soubory jsou na místě!
✅ Dashboard generated: dashboard.html
✅ dashboard.html ready
✅ Dashboard generated: dashboard.html
✅ dashboard.html ready
